In [ ]:
# options(timeout = 300)
# install.packages("glue")
# install.packages("dplyr")

Устанавливаю пакет в 'C:/Users/Polina/AppData/Local/R/win-library/4.6'
(потому что 'lib' не определено)



package 'dplyr' successfully unpacked and MD5 sums checked

Скачанные бинарные пакеты находятся в
	C:\Users\Polina\AppData\Local\Temp\RtmpiEoS8h\downloaded_packages


In [26]:
library(glue)
library(dplyr)


Присоединяю пакет: 'dplyr'


Следующие объекты скрыты от 'package:stats':

    filter, lag


Следующие объекты скрыты от 'package:base':

    intersect, setdiff, setequal, union




### 2.1.3. Предварительная обработка данных (заполнение пропусков, фильтрация, преобразование).

In [4]:
df <- read.csv('data/Sleep_health_and_lifestyle_dataset.csv')
head(df)

,Person.ID,Gender,Age,Occupation,Sleep.Duration,Quality.of.Sleep,Physical.Activity.Level,Stress.Level,BMI.Category,Blood.Pressure,Heart.Rate,Daily.Steps,Sleep.Disorder
,<int>,<chr>,<int>,<chr>,<dbl>,<int>,<int>,<int>,<chr>,<chr>,<int>,<int>,<chr>
1,1,Male,27,Software Engineer,6.1,6,42,6,Overweight,126/83,77,4200,None
2,2,Male,28,Doctor,6.2,6,60,8,Normal,125/80,75,10000,None
3,3,Male,28,Doctor,6.2,6,60,8,Normal,125/80,75,10000,None
4,4,Male,28,Sales Representative,5.9,4,30,8,Obese,140/90,85,3000,Sleep Apnea
5,5,Male,28,Sales Representative,5.9,4,30,8,Obese,140/90,85,3000,Sleep Apnea
6,6,Male,28,Software Engineer,5.9,4,30,8,Obese,140/90,85,3000,Insomnia


In [16]:
str(df)

'data.frame':	374 obs. of  13 variables:
 $ Person.ID              : int  1 2 3 4 5 6 7 8 9 10 ...
 $ Gender                 : chr  "Male" "Male" "Male" "Male" ...
 $ Age                    : int  27 28 28 28 28 28 29 29 29 29 ...
 $ Occupation             : chr  "Software Engineer" "Doctor" "Doctor" "Sales Representative" ...
 $ Sleep.Duration         : num  6.1 6.2 6.2 5.9 5.9 5.9 6.3 7.8 7.8 7.8 ...
 $ Quality.of.Sleep       : int  6 6 6 4 4 4 6 7 7 7 ...
 $ Physical.Activity.Level: int  42 60 60 30 30 30 40 75 75 75 ...
 $ Stress.Level           : int  6 8 8 8 8 8 7 6 6 6 ...
 $ BMI.Category           : chr  "Overweight" "Normal" "Normal" "Obese" ...
 $ Blood.Pressure         : chr  "126/83" "125/80" "125/80" "140/90" ...
 $ Heart.Rate             : int  77 75 75 85 85 85 82 70 70 70 ...
 $ Daily.Steps            : int  4200 10000 10000 3000 3000 3000 3500 8000 8000 8000 ...
 $ Sleep.Disorder         : chr  "None" "None" "None" "Sleep Apnea" ...


In [6]:
colSums(is.na(df))

Person.ID                  Gender                     Age 
                      0                       0                       0 
             Occupation          Sleep.Duration        Quality.of.Sleep 
                      0                       0                       0 
Physical.Activity.Level            Stress.Level            BMI.Category 
                      0                       0                       0 
         Blood.Pressure              Heart.Rate             Daily.Steps 
                      0                       0                       0 
         Sleep.Disorder 
                      0

Пропусков не обнаружено

In [13]:
string_columns <- c('Gender', 'Occupation', 'BMI.Category', 'Blood.Pressure', 'Sleep.Disorder')
for (col in string_columns) {
    output <- paste(unique(df[[col]]), collapse = ', ')
    print(glue('Столбец {col}: {output}'))
}


Столбец Gender: Male, Female
Столбец Occupation: Software Engineer, Doctor, Sales Representative, Teacher, Nurse, Engineer, Accountant, Scientist, Lawyer, Salesperson, Manager
Столбец BMI.Category: Overweight, Normal, Obese, Normal Weight
Столбец Blood.Pressure: 126/83, 125/80, 140/90, 120/80, 132/87, 130/86, 117/76, 118/76, 128/85, 131/86, 128/84, 115/75, 135/88, 129/84, 130/85, 115/78, 119/77, 121/79, 125/82, 135/90, 122/80, 142/92, 140/95, 139/91, 118/75
Столбец Sleep.Disorder: None, Sleep Apnea, Insomnia


Occupation - много.выбрать в категории \
BMI.Category - дубликат \
Blood.Pressure в число и группы (потом)

In [29]:
df$BMI.Category <- recode(df$BMI.Category, 'Normal Weight' = 'Normal')

In [33]:
df$Occupation.Category <- df$Occupation

df$Occupation.Category[df$Occupation.Category %in% c("Nurse", "Doctor")] <- 'Healthcare'
df$Occupation.Category[df$Occupation.Category %in% c("Software Engineer", "Engineer", "Scientist", "Accountant")] <- 'Office'
df$Occupation.Category[df$Occupation.Category %in% c('Salesperson', 'Sales Representative', 'Manager', 'Lawyer', 'Teacher')] <- 'Social'

In [35]:
splitted_blood_pressure <- strsplit(df$Blood.Pressure, '/')
df$Systolic <- as.numeric(sapply(splitted_blood_pressure, '[', 1))
df$Diastolic <- as.numeric(sapply(splitted_blood_pressure, '[', 2))

In [36]:
str(df)

'data.frame':	374 obs. of  16 variables:
 $ Person.ID              : int  1 2 3 4 5 6 7 8 9 10 ...
 $ Gender                 : chr  "Male" "Male" "Male" "Male" ...
 $ Age                    : int  27 28 28 28 28 28 29 29 29 29 ...
 $ Occupation             : chr  "Software Engineer" "Doctor" "Doctor" "Sales Representative" ...
 $ Sleep.Duration         : num  6.1 6.2 6.2 5.9 5.9 5.9 6.3 7.8 7.8 7.8 ...
 $ Quality.of.Sleep       : int  6 6 6 4 4 4 6 7 7 7 ...
 $ Physical.Activity.Level: int  42 60 60 30 30 30 40 75 75 75 ...
 $ Stress.Level           : int  6 8 8 8 8 8 7 6 6 6 ...
 $ BMI.Category           : chr  "Overweight" "Normal" "Normal" "Obese" ...
 $ Blood.Pressure         : chr  "126/83" "125/80" "125/80" "140/90" ...
 $ Heart.Rate             : int  77 75 75 85 85 85 82 70 70 70 ...
 $ Daily.Steps            : int  4200 10000 10000 3000 3000 3000 3500 8000 8000 8000 ...
 $ Sleep.Disorder         : chr  "None" "None" "None" "Sleep Apnea" ...
 $ Occupation.Category    : chr  "Of